<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


<h1>Lab: Implement and Test an LSTM-Based Classifier in PyTorch</h1> 


Estimated time needed: **30** minutes


<h2>Learning Objectives</h2>
In this lab, you will:
<ul><li> Prepare and preprocess text data for LSTM-based sequence classification</li><li> Implement an LSTM model architecture using PyTorch modules</li><li> Train the LSTM classifier using appropriate loss functions and optimizers</li><li> Evaluate model performance using metrics such as accuracy, precision, and recall</li><li> Identify model behaviors such as overfitting, underfitting, and convergence trends</li></ul> 


<h2>Table of Contents</h2>
<p>In this lab, you will build, train, and evaluate an LSTM-based classifier for sequence classification using PyTorch.</p>

<ol>
<li><a href="#Import-Required-Libraries">Import Required Libraries</a> </li>
<li><a href="#Long-Short-Term-Memory-(LSTM)-Networks-for-Text-Classification">Long Short-Term Memory (LSTM) Networks for Text Classification</a> </li>
<li><a href="#Create-Text-Classification-Data">Create Text Classification Data</a> </li>
<li><a href="#Preprocess-Text-Data">Preprocess Text Data</a> </li>
<li><a href="#Create-a-Dataset-and-DataLoader">Create a Dataset and DataLoader</a> </li>
<li><a href="#Implement-an-LSTM-Classifier">Implement an LSTM Classifier</a> </li>
<li><a href="#Train-the-Classifier">Train the Classifier</a> </li>
<li><a href="#Evaluate-Model-Performance">Evaluate Model Performance</a> </li>
<li><a href="#Analyze-Model-Behavior">Analyze Model Behavior</a> </li>

</ol>

<hr>


## Import Required Libraries


The following required libraries are __not__ pre-installed in the Skills Network Labs environment. __You will need to run the following cells__ to install or import them:


In [1]:
# Install PyTorch

!pip install torch

In [2]:
# Import the libraries we need for this lab

import random
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from torch.utils.data import Dataset, DataLoader, random_split

random.seed(2)
torch.manual_seed(2)

/opt/conda/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))

<!--Empty Space for separating topics-->


## Long Short-Term Memory (LSTM) Networks for Text Classification


Long Short-Term Memory networks are recurrent neural networks designed to model ordered data. In text classification, an LSTM reads token embeddings over time and summarizes the sequence into hidden states used for prediction.


<p style="text-align:center">
    <img src="https://commons.wikimedia.org/wiki/Special:Redirect/file/The%20LSTM%20Cell.svg" width="650" alt="LSTM cell architecture diagram">
</p>
<p style="text-align:center"><em>Source: Wikimedia Commons, The LSTM Cell by Guillaume Chevalier</em></p>


The gates in an LSTM help control what information is stored, forgotten, and passed forward through the sequence.


<!--Empty Space for separating topics-->


## Create Text Classification Data


Create a small sentiment-style text dataset. The label <code>1</code> represents positive sentiment, and the label <code>0</code> represents negative sentiment:


In [3]:
# Create a small text classification dataset

positive_sentences = [
    "the product works great and feels reliable",
    "the service was fast helpful and friendly",
    "the lesson was clear practical and useful",
    "the headphones sound excellent and comfortable",
    "the app is responsive stable and easy to use",
    "the package arrived early and in perfect condition",
    "the support team fixed the issue quickly",
    "the course examples were relevant and engaging",
    "the device feels durable and performs well",
    "the interface is clean polished and intuitive",
]

negative_sentences = [
    "the product broke quickly and feels cheap",
    "the service was slow rude and unhelpful",
    "the lesson was confusing repetitive and unclear",
    "the headphones disconnect often and sound terrible",
    "the app is unstable slow and hard to use",
    "the package arrived late and badly damaged",
    "the support team ignored the issue completely",
    "the course examples were outdated and confusing",
    "the device feels fragile and performs poorly",
    "the interface is messy awkward and frustrating",
]

data = [(sentence, 1) for sentence in positive_sentences] + [(sentence, 0) for sentence in negative_sentences]
random.shuffle(data)

print("Number of examples:", len(data))
print(data[:3])

Number of examples: 20
[('the course examples were relevant and engaging', 1), ('the support team fixed the issue quickly', 1), ('the course examples were outdated and confusing', 0)]

Expand the dataset with simple sentence variations so the LSTM has more examples to train on:


In [4]:
# Create simple text variations

prefixes = ["", "overall ", "today ", "in my experience "]
expanded_data = []

for sentence, label in data:
    for prefix in prefixes:
        expanded_data.append((prefix + sentence, label))

random.shuffle(expanded_data)

print("Expanded examples:", len(expanded_data))
print(expanded_data[:3])

Expanded examples: 80
[('the course examples were relevant and engaging', 1), ('today the lesson was clear practical and useful', 1), ('the package arrived early and in perfect condition', 1)]

<!--Empty Space for separating topics-->


## Preprocess Text Data


Preprocessing converts raw text into ordered token sequences. These sequences preserve word order, which is important for recurrent models:


In [5]:
# Create a tokenizer

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

Tokenize all examples and build a vocabulary:


In [6]:
# Build tokenized examples and vocabulary

tokenized_examples = [(tokenize(text), label) for text, label in expanded_data]
token_counts = Counter(token for tokens, label in tokenized_examples for token in tokens)

special_tokens = ["<pad>", "<unk>"]
vocab = {token: index for index, token in enumerate(special_tokens)}

for token in token_counts:
    if token not in vocab:
        vocab[token] = len(vocab)

pad_id = vocab["<pad>"]
unk_id = vocab["<unk>"]

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:10])

Vocabulary size: 81
[('<pad>', 0), ('<unk>', 1), ('the', 2), ('course', 3), ('examples', 4), ('were', 5), ('relevant', 6), ('and', 7), ('engaging', 8), ('today', 9)]

Convert tokens to token IDs:


In [7]:
# Convert tokens to token IDs

def encode_tokens(tokens, vocabulary):
    return [vocabulary.get(token, unk_id) for token in tokens]

encoded_examples = [(encode_tokens(tokens, vocab), label) for tokens, label in tokenized_examples]

print(tokenized_examples[0])
print(encoded_examples[0])

(['the', 'course', 'examples', 'were', 'relevant', 'and', 'engaging'], 1)
([2, 3, 4, 5, 6, 7, 8], 1)

<!--Empty Space for separating topics-->


## Create a Dataset and DataLoader


Create a custom PyTorch Dataset that returns one encoded sequence and one label at a time:


In [8]:
# Create a custom Dataset

class TextClassificationDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        token_ids, label = self.examples[index]
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(label, dtype=torch.float32)


full_dataset = TextClassificationDataset(encoded_examples)

print("Dataset size:", len(full_dataset))
print(full_dataset[0])

Dataset size: 80
(tensor([2, 3, 4, 5, 6, 7, 8]), tensor(1.))

Split the dataset into training and validation sets:


In [9]:
# Split the dataset

train_size = int(0.8 * len(full_dataset))
valid_size = len(full_dataset) - train_size

train_dataset, valid_dataset = random_split(
    full_dataset,
    [train_size, valid_size],
    generator=torch.Generator().manual_seed(2),
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(valid_dataset))

Training examples: 64
Validation examples: 16

Create a collate function that pads variable-length sequences in each batch:


In [10]:
# Create a collate function for LSTM batches

def collate_batch(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(sequence) for sequence in sequences], dtype=torch.long)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=pad_id)
    labels = torch.stack(labels)
    return padded_sequences, labels, lengths

Create DataLoaders for training and validation:


In [11]:
# Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(valid_dataset, batch_size=8, shuffle=False, collate_fn=collate_batch)

batch_sequences, batch_labels, batch_lengths = next(iter(train_loader))

print("Batch sequences:", batch_sequences.shape)
print("Batch labels:", batch_labels.shape)
print("Batch lengths:", batch_lengths)

Batch sequences: torch.Size([8, 10])
Batch labels: torch.Size([8])
Batch lengths: tensor([ 7,  8,  8,  8,  8,  8,  8, 10])

<!--Empty Space for separating topics-->


## Implement an LSTM Classifier


The model uses an embedding layer to represent tokens, an LSTM to process the sequence, and a linear layer to produce one binary classification logit:


In [12]:
# Create an LSTM classifier

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, padding_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, sequences, lengths):
        embedded = self.embedding(sequences)
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, (hidden, cell) = self.lstm(packed)
        logits = self.fc(hidden[-1])
        return logits.squeeze(1)

Instantiate the model and inspect its output shape for one batch:


In [13]:
# Instantiate the LSTM classifier

model = LSTMClassifier(
    vocab_size=len(vocab),
    embedding_dim=32,
    hidden_dim=32,
    output_dim=1,
    padding_idx=pad_id,
)

sample_logits = model(batch_sequences, batch_lengths)

print(model)
print("Output shape:", sample_logits.shape)

LSTMClassifier(
  (embedding): Embedding(81, 32, padding_idx=0)
  (lstm): LSTM(32, 32, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)
Output shape: torch.Size([8])

<!--Empty Space for separating topics-->


## Train the Classifier


Use binary cross-entropy with logits for binary classification and the Adam optimizer for training:


In [14]:
# Define the loss function and optimizer

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

Create helper functions for one training epoch and one evaluation pass:


In [15]:
# Create training and evaluation helpers

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for sequences, labels, lengths in loader:
        optimizer.zero_grad()
        logits = model(sequences, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for sequences, labels, lengths in loader:
            logits = model(sequences, lengths)
            loss = criterion(logits, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

Train the model for a small number of epochs and track convergence:


In [17]:
# Train the LSTM classifier

history = []
epochs = 8

for epoch in range(epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
    valid_loss = evaluate_loss(model, valid_loader, criterion)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "valid_loss": valid_loss})
    print(f"Epoch {epoch + 1}: train loss={train_loss:.4f}, valid loss={valid_loss:.4f}")

Epoch 1: train loss=0.6726, valid loss=0.5687Epoch 2: train loss=0.4586, valid loss=0.3067Epoch 3: train loss=0.1663, valid loss=0.0563Epoch 4: train loss=0.0224, valid loss=0.0072Epoch 5: train loss=0.0038, valid loss=0.0019Epoch 6: train loss=0.0012, valid loss=0.0009Epoch 7: train loss=0.0007, valid loss=0.0007Epoch 8: train loss=0.0006, valid loss=0.0005

<!--Empty Space for separating topics-->


## Evaluate Model Performance


Evaluate predictions using accuracy, precision, and recall. Precision measures how many predicted positive examples were correct, while recall measures how many true positive examples were found:


In [16]:
# Generate predictions

def collect_predictions(model, loader):
    model.eval()
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for sequences, labels, lengths in loader:
            logits = model(sequences, lengths)
            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= 0.5).long()
            all_predictions.extend(predictions.tolist())
            all_labels.extend(labels.long().tolist())
    return all_predictions, all_labels

valid_predictions, valid_labels = collect_predictions(model, valid_loader)

print("Predictions:", valid_predictions)
print("Labels:", valid_labels)

Predictions: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Labels: [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1]

Compute accuracy, precision, and recall:


In [18]:
# Calculate evaluation metrics

def classification_metrics(predictions, labels):
    tp = sum(1 for pred, label in zip(predictions, labels) if pred == 1 and label == 1)
    tn = sum(1 for pred, label in zip(predictions, labels) if pred == 0 and label == 0)
    fp = sum(1 for pred, label in zip(predictions, labels) if pred == 1 and label == 0)
    fn = sum(1 for pred, label in zip(predictions, labels) if pred == 0 and label == 1)

    accuracy = (tp + tn) / max(len(labels), 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    return {"accuracy": accuracy, "precision": precision, "recall": recall}

metrics = classification_metrics(valid_predictions, valid_labels)

metrics

{'accuracy': 0.5625, 'precision': 0.5625, 'recall': 1.0}

<!--Empty Space for separating topics-->


## Analyze Model Behavior


Training and validation loss trends help identify convergence, overfitting, and underfitting. Loss should generally decrease as the model learns:


In [19]:
# Inspect convergence history

for row in history:
    print(row)

{'epoch': 1, 'train_loss': 0.672623299062252, 'valid_loss': 0.5687375664710999}
{'epoch': 2, 'train_loss': 0.45861320197582245, 'valid_loss': 0.3067011535167694}
{'epoch': 3, 'train_loss': 0.16625168174505234, 'valid_loss': 0.05626205727458}
{'epoch': 4, 'train_loss': 0.022410315228626132, 'valid_loss': 0.007245506625622511}
{'epoch': 5, 'train_loss': 0.0038108636217657477, 'valid_loss': 0.0019391928217373788}
{'epoch': 6, 'train_loss': 0.0012283802861929871, 'valid_loss': 0.000945250503718853}
{'epoch': 7, 'train_loss': 0.0007274710733327083, 'valid_loss': 0.0006543521594721824}
{'epoch': 8, 'train_loss': 0.0005599035357590765, 'valid_loss': 0.000525289069628343}

Create a simple rule-based interpretation of the loss trend:


In [20]:
# Identify model behavior from loss trends

first_train_loss = history[0]["train_loss"]
last_train_loss = history[-1]["train_loss"]
first_valid_loss = history[0]["valid_loss"]
last_valid_loss = history[-1]["valid_loss"]

if last_train_loss < first_train_loss and last_valid_loss <= first_valid_loss:
    behavior = "The model shows convergence because both training and validation loss improved."
elif last_train_loss < first_train_loss and last_valid_loss > first_valid_loss:
    behavior = "The model may be overfitting because training loss improved while validation loss increased."
else:
    behavior = "The model may be underfitting because the training loss did not improve enough."

print(behavior)

The model shows convergence because both training and validation loss improved.


<a href="https://dataplatform.cloud.ibm.com/registration/stepone?utm_source=skills_network&utm_content=in_lab_content_link&utm_id=Lab-IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork&context=cpdaas&apps=data_science_experience%2Cwatson_machine_learning"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0110EN-SkillsNetwork/Template/module%201/images/Watson_Studio.png"></a>


<!--Empty Space for separating topics-->


<h2>About the Author:</h2> 

**Harish Pant** is an accomplished IT Consultant specializing in bridging the gap between complex AI technologies and practical business applications. Holding a Bachelor’s degree in Computer Science with a strong background in Machine Learning, Harish serves clients across diverse industries—including fintech, retail, and manufacturing—helping them navigate AI adoption. Harish is known for translating technical AI concepts into actionable business strategies and delivering engaging, high-impact presentations.


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2026-05-11  | 1.0  | IBM Skills Network  |  Created lab for implementing and testing an LSTM-based classifier in PyTorch |
-->


<hr>





## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>
